In [1]:
# Imports, device, and seed

import os, random, math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, BatchNorm
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from scipy.sparse import issparse
from pathlib import Path
from joblib import dump, load

# Reproducibility
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
# Utility helpers
def make_undirected(edge_index_np):
    rows, cols = edge_index_np
    mask = rows != cols
    rows, cols = rows[mask], cols[mask]
    rows2 = np.concatenate([rows, cols])
    cols2 = np.concatenate([cols, rows])
    edges = np.vstack([rows2, cols2]).T
    edges = np.unique(edges, axis=0)
    return edges.T

def split_masks(n, train=0.7, val=0.15, test=0.15, seed=42):
    idx = np.random.RandomState(seed).permutation(n)
    n_train = int(train*n); n_val = int(val*n)
    train_idx = idx[:n_train]
    val_idx   = idx[n_train:n_train+n_val]
    test_idx  = idx[n_train+n_val:]
    masks = {}
    for name, idc in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
        m = torch.zeros(n, dtype=torch.bool)
        m[idc] = True
        masks[name] = m
    return masks

In [3]:
# Load dataset

CSV_PATH = "Concrete_Data.csv"          
TARGET_COL = "Concrete compressive strength(MPa, megapascals) "      

df = pd.read_csv(CSV_PATH)
assert TARGET_COL in df.columns, f"Target missing; check column names: {df.columns.tolist()}"
print("Dataset:", df.shape)

Dataset: (1030, 9)


In [4]:
def make_X(df, mode="A"):
    if mode == "A":
        cols = [
            "Cement (component 1)(kg in a m^3 mixture)",
            "Blast Furnace Slag (component 2)(kg in a m^3 mixture)",
            "Fly Ash (component 3)(kg in a m^3 mixture)",
            "Water  (component 4)(kg in a m^3 mixture)",
            "Superplasticizer (component 5)(kg in a m^3 mixture)",
            "Coarse Aggregate  (component 6)(kg in a m^3 mixture)",
            "Fine Aggregate (component 7)(kg in a m^3 mixture)",
            "Age (day)"
        ]
        return df[cols].copy()

    if mode == "B":
        cols = [
            "Cement (component 1)(kg in a m^3 mixture)",
            "Water  (component 4)(kg in a m^3 mixture)",
            "Superplasticizer (component 5)(kg in a m^3 mixture)",
            "Age (day)"
        ]
        return df[cols].copy()

    if mode == "C":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"
        tmp = df[[c_cem,c_slag,c_fly,c_wat,c_sp,c_ca,c_fa,c_age]].copy()
        tmp["Binder"]    = tmp[c_cem] + tmp[c_slag] + tmp[c_fly]
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
        tmp["Fluid"]     = tmp[c_wat] + tmp[c_sp]
        eps = 1e-9
        tmp["w_b"]   = tmp[c_wat] / (tmp["Binder"] + eps)
        tmp["agg_b"] = tmp["Aggregate"] / (tmp["Binder"] + eps)
        tmp["sp_b"]  = tmp[c_sp] / (tmp["Binder"] + eps)
        return tmp[["Binder","Aggregate","Fluid","w_b","agg_b","sp_b", c_age]].copy()

    if mode == "D":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"

        tmp = df[[c_cem, c_slag, c_fly, c_wat, c_sp, c_ca, c_fa, c_age]].copy()
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
    
        return tmp[[c_cem, c_slag, c_fly, c_wat, c_sp, "Aggregate", c_age]].copy()


    if mode == "E":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"
    
        tmp = df[[c_cem, c_slag, c_fly, c_wat, c_sp, c_ca, c_fa, c_age]].copy()
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
        tmp["SEM"] = tmp[c_slag] + tmp[c_fly]
    
        return tmp[[c_cem, c_wat, c_sp, "Aggregate", "SEM", c_age]].copy()


In [5]:
# Split features/target

y_raw = df[TARGET_COL].to_numpy().reshape(-1, 1)
X_df = make_X(df, mode="E")   
print("Using columns:", list(X_df.columns))

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]

def build_ohe():
    try:

        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", build_ohe())
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop",
    sparse_threshold=0.3  
)

X = preprocess.fit_transform(X_df)
if issparse(X):
    X = X.toarray()


from sklearn.preprocessing import StandardScaler as _SS
y_scaler = _SS().fit(y_raw)
y = y_scaler.transform(y_raw).ravel()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Numeric cols:", len(num_cols), "| Categorical cols:", len(cat_cols))

Using columns: ['Cement (component 1)(kg in a m^3 mixture)', 'Water  (component 4)(kg in a m^3 mixture)', 'Superplasticizer (component 5)(kg in a m^3 mixture)', 'Aggregate', 'SEM', 'Age (day)']
X shape: (1030, 6) | y shape: (1030,)
Numeric cols: 6 | Categorical cols: 0


In [6]:
# Build a kNN graph in feature space (undirected)

k = 5 
nbrs = NearestNeighbors(n_neighbors=min(k+1, X.shape[0]), metric="euclidean")
nbrs.fit(X)
A = nbrs.kneighbors_graph(X, n_neighbors=min(k+1, X.shape[0]), mode="connectivity")
A = A.maximum(A.T)

rows, cols = A.nonzero()
edge_index_np = np.vstack([rows, cols])
edge_index_np = make_undirected(edge_index_np)

edge_index = torch.tensor(edge_index_np, dtype=torch.long)
x = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

data = Data(x=x, edge_index=edge_index, y=y_t)
data

Data(x=[1030, 6], edge_index=[2, 6554], y=[1030])

In [7]:
#Train/Val/Test node masks

masks = split_masks(data.num_nodes, train=0.70, val=0.15, test=0.15, seed=42)
data.train_mask = masks["train"]
data.val_mask = masks["val"]
data.test_mask = masks["test"]
{ k: int(v.sum().item()) for k, v in masks.items() }

{'train': 721, 'val': 154, 'test': 155}

In [8]:
# Define a baseline GNN (GraphSAGE for regression)

class SAGENet(nn.Module):
    def __init__(self, in_channels, hidden=128, layers=3, dropout=0.25, aggr="mean"):
        super().__init__()
        assert aggr in {"mean", "max", "add"}, "aggr must be 'mean', 'max', or 'add'"

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        self.convs.append(SAGEConv(in_channels, hidden, aggr=aggr))
        self.bns.append(BatchNorm(hidden))

        for _ in range(layers - 2):
            self.convs.append(SAGEConv(hidden, hidden, aggr=aggr))
            self.bns.append(BatchNorm(hidden))

        self.out_lin = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out_lin(x).view(-1)


model = SAGENet(in_channels=data.num_features, hidden=128, layers=3, dropout=0.25, aggr="mean").to(device)
data = data.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

In [9]:
# Train with early stopping
# save to a local folder that we create explicitly

CKPT_DIR = Path.cwd() / "artifacts"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_path = CKPT_DIR / "gnn_best.pt"

best_val = float("inf")
patience, bad = 30, 0
history = []

for epoch in range(1, 1001):
    model.train()
    optimizer.zero_grad()
    pred = model(data.x, data.edge_index)
    loss = criterion(pred[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_all = model(data.x, data.edge_index)
        val_loss = criterion(pred_all[data.val_mask],  data.y[data.val_mask]).item()
        test_loss = criterion(pred_all[data.test_mask], data.y[data.test_mask]).item()
    history.append((epoch, float(loss.item()), val_loss, test_loss))

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        bad = 0
        try:
            tmp_path = CKPT_DIR / f"tmp_epoch{epoch}.pt"
            torch.save(model.state_dict(), tmp_path.open("wb"))
            tmp_path.replace(best_path)  
        except Exception as e:
            fb = CKPT_DIR / f"gnn_best_epoch{epoch}.pt"
            torch.save(model.state_dict(), fb.open("wb"))
            print(f"[WARN] Could not write {best_path.name}: {e}. Saved fallback: {fb.name}")
    else:
        bad += 1

    if epoch % 25 == 0 or bad == 0:
        print(f"Epoch {epoch:4d} | train {loss.item():.4f} | val {val_loss:.4f} | test {test_loss:.4f}")
    if bad >= patience:
        print(f"Early stop at epoch {epoch}. Best val {best_val:.4f}")
        break

# Load best
if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f"Loaded best checkpoint from: {best_path}")
else:
    print("[WARN] No best checkpoint found; using last-epoch weights.")

Epoch    1 | train 1.3408 | val 0.7859 | test 0.8242
Epoch    2 | train 0.8749 | val 0.6397 | test 0.6980
Epoch    3 | train 0.6528 | val 0.5392 | test 0.6132
Epoch    4 | train 0.5852 | val 0.4731 | test 0.5611
Epoch    5 | train 0.5453 | val 0.4347 | test 0.5306
Epoch    6 | train 0.5466 | val 0.4114 | test 0.5088
Epoch    7 | train 0.4968 | val 0.3970 | test 0.4928
Epoch    8 | train 0.5085 | val 0.3915 | test 0.4842
Epoch    9 | train 0.4481 | val 0.3907 | test 0.4779
Epoch   15 | train 0.4060 | val 0.3874 | test 0.4206
Epoch   16 | train 0.3934 | val 0.3677 | test 0.3995
Epoch   17 | train 0.3684 | val 0.3436 | test 0.3780
Epoch   18 | train 0.3620 | val 0.3171 | test 0.3559
Epoch   19 | train 0.3309 | val 0.2917 | test 0.3348
Epoch   20 | train 0.3388 | val 0.2693 | test 0.3175
Epoch   21 | train 0.3063 | val 0.2518 | test 0.3049
Epoch   22 | train 0.3099 | val 0.2391 | test 0.2954
Epoch   23 | train 0.3030 | val 0.2300 | test 0.2876
Epoch   24 | train 0.3174 | val 0.2240 | test 

In [10]:
# Evaluate in original units

import torch

@torch.no_grad()
def evaluate_split(mask, name):
    model.eval()
    pred_scaled = model(data.x, data.edge_index).detach().cpu().numpy().reshape(-1, 1)
    y_scaled    = data.y.detach().cpu().numpy().reshape(-1, 1)

    # invert target scaling
    y_true = y_scaler.inverse_transform(y_scaled).ravel()
    y_pred = y_scaler.inverse_transform(pred_scaled).ravel()
    m = mask.cpu().numpy()

    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    rmse = root_mean_squared_error(y_true[m], y_pred[m])
    mae  = mean_absolute_error(y_true[m], y_pred[m])
    r2   = r2_score(y_true[m], y_pred[m])
    mape = np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100
    print(f"{name:5s} | RMSE {rmse:.4f} | MAE {mae:.4f} | R² {r2:.4f} | MAPE {mape:.2f}%")
    return rmse, mae, r2, mape

evaluate_split(data.train_mask, "Train")
evaluate_split(data.val_mask,   "Val")
evaluate_split(data.test_mask,  "Test")

Train | RMSE 4.3908 | MAE 3.3317 | R² 0.9343 | MAPE 11.81%
Val   | RMSE 5.3758 | MAE 4.1202 | R² 0.8809 | MAPE 14.42%
Test  | RMSE 5.1540 | MAE 3.9955 | R² 0.8918 | MAPE 15.08%


(5.1540346, 3.9955165, 0.891834557056427, 15.07813036441803)

In [11]:
import sys
import torch
import torch_geometric
import sklearn

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.7.1+cpu
PyTorch Geometric: 2.8.0.post1
scikit-learn: 1.5.1
